# Blinkit Grocery — Data Understanding, Quality Audit & Cleaning

**Steps covered:**
1. Understand the data (structure, types, missing values, duplicates)
2. Data quality report (audit every issue)
3. Data cleaning (apply all fixes, export clean file for Power BI)

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

FILE = 'Blinkit Grocery.xlsx'

C:\Users\m21si\AppData\Local\Temp\ipykernel_23192\3625547946.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


## STEP 1 — Understand the Data

In [2]:
# List every sheet in the workbook
xl = pd.ExcelFile(FILE)
print('Sheets:', xl.sheet_names)

df = xl.parse(xl.sheet_names[0])
print('Shape:', df.shape)
df.head()

Sheets: ['Tableau BlinkIT Grocery Project']


Shape: (8523, 12)


,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [3]:
# Column names, data types, missing values, uniques — one overview table
overview = pd.DataFrame({
    'dtype': df.dtypes,
    'missing': df.isna().sum(),
    'missing_%': (df.isna().mean() * 100).round(1),
    'unique': df.nunique(),
})
overview

,dtype,missing,missing_%,unique
Item_Identifier,object,0,0.0,1559
Item_Weight,float64,1463,17.2,415
Item_Fat_Content,object,0,0.0,5
Item_Visibility,float64,0,0.0,7880
Item_Type,object,0,0.0,16
Item_MRP,float64,0,0.0,5938
Outlet_Identifier,object,0,0.0,10
Outlet_Establishment_Year,int64,0,0.0,9
Outlet_Size,object,2410,28.3,3
Outlet_Location_Type,object,0,0.0,3


In [4]:
# Duplicates
print('Full duplicate rows:', df.duplicated().sum())
print('Duplicate (Item, Outlet) pairs:', df.duplicated(['Item_Identifier', 'Outlet_Identifier']).sum())

Full duplicate rows: 0
Duplicate (Item, Outlet) pairs: 0


In [5]:
# Unique values of every categorical column
for col in df.select_dtypes('object').columns.drop('Item_Identifier'):
    print(f'{col}: {df[col].value_counts(dropna=False).to_dict()}\n')

Item_Fat_Content: {'Low Fat': 5089, 'Regular': 2889, 'LF': 316, 'reg': 117, 'low fat': 112}

Item_Type: {'Fruits and Vegetables': 1232, 'Snack Foods': 1200, 'Household': 910, 'Frozen Foods': 856, 'Dairy': 682, 'Canned': 649, 'Baking Goods': 648, 'Health and Hygiene': 520, 'Soft Drinks': 445, 'Meat': 425, 'Breads': 251, 'Hard Drinks': 214, 'Others': 169, 'Starchy Foods': 148, 'Breakfast': 110, 'Seafood': 64}

Outlet_Identifier: {'OUT027': 935, 'OUT013': 932, 'OUT049': 930, 'OUT046': 930, 'OUT035': 930, 'OUT045': 929, 'OUT018': 928, 'OUT017': 926, 'OUT010': 555, 'OUT019': 528}

Outlet_Size: {'Medium': 2793, nan: 2410, 'Small': 2388, 'High': 932}

Outlet_Location_Type: {'Tier 3': 3350, 'Tier 2': 2785, 'Tier 1': 2388}

Outlet_Type: {'Supermarket Type1': 5577, 'Grocery Store': 1083, 'Supermarket Type3': 935, 'Supermarket Type2': 928}



In [6]:
# Numeric summary
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Item_Weight,7060.0,12.857645,4.643456,4.555,8.773750,12.600000,16.850000,21.350000
Item_Visibility,8523.0,0.066132,0.051598,0.000,0.026989,0.053931,0.094585,0.328391
Item_MRP,8523.0,140.992782,62.275067,31.290,93.826500,143.012800,185.643700,266.888400
Outlet_Establishment_Year,8523.0,1997.831867,8.371760,1985.000,1987.000000,1999.000000,2004.000000,2009.000000
Item_Outlet_Sales,8523.0,2181.288914,1706.499616,33.290,834.247400,1794.331000,3101.296400,13086.964800


## STEP 2 — Data Quality Report

| # | Issue | Fix |
|---|-------|-----|
| 1 | `Item_Weight` missing 1,463 rows (17.2%) — only outlets OUT027 & OUT019 (systematic gap) | Impute per-SKU weight from other outlets; fallback = Item_Type mean |
| 2 | `Outlet_Size` missing 2,410 rows — exactly OUT010, OUT017, OUT045 | Fill with 'Small' (business rule) |
| 3 | `Item_Fat_Content` inconsistent labels: LF / low fat / reg | Standardize to Low Fat / Regular |
| 4 | All 1,599 NC (non-consumable) items tagged 'Low Fat' — impossible | Recode to 'Non-Edible' |
| 5 | 526 rows with `Item_Visibility = 0` but positive sales — impossible | Treat 0 as missing, impute per-SKU mean |
| 6 | 186 sales outliers (> Q3 + 1.5×IQR) | Keep — genuine high performers, not errors |

In [7]:
# Verify each issue with code

# 1. Missing weight — which outlets?
print('Weight missing by outlet:', df[df.Item_Weight.isna()].Outlet_Identifier.value_counts().to_dict())

# 2. Missing size — which outlets?
print('Size missing by outlet:  ', df[df.Outlet_Size.isna()].Outlet_Identifier.value_counts().to_dict())

# 3. Fat label variants
print('Fat labels:', df.Item_Fat_Content.value_counts().to_dict())

# 4. NC items with fat labels
nc = df[df.Item_Identifier.str[:2] == 'NC']
print('NC items fat labels:', nc.Item_Fat_Content.value_counts().to_dict(), f'({len(nc)} rows)')

# 5. Zero visibility with sales
print('Visibility == 0 rows:', (df.Item_Visibility == 0).sum(),
      '| all have sales > 0:', bool((df.loc[df.Item_Visibility == 0, 'Item_Outlet_Sales'] > 0).all()))

# 6. Outliers
q1, q3 = df.Item_Outlet_Sales.quantile([.25, .75])
print('Sales outliers (> Q3 + 1.5*IQR):', (df.Item_Outlet_Sales > q3 + 1.5 * (q3 - q1)).sum())

# Sanity: no negatives, no blank strings
print('Negative values:', (df.select_dtypes('number') < 0).sum().sum())
print('Blank strings:  ', df.select_dtypes('object').apply(lambda c: c.astype(str).str.strip().eq('').sum()).sum())

Weight missing by outlet: {'OUT027': 935, 'OUT019': 528}
Size missing by outlet:   {'OUT045': 929, 'OUT017': 926, 'OUT010': 555}
Fat labels: {'Low Fat': 5089, 'Regular': 2889, 'LF': 316, 'reg': 117, 'low fat': 112}
NC items fat labels: {'Low Fat': 1477, 'LF': 94, 'low fat': 28} (1599 rows)
Visibility == 0 rows: 526 | all have sales > 0: True
Sales outliers (> Q3 + 1.5*IQR): 186
Negative values: 0
Blank strings:   0


## STEP 3 — Data Cleaning

In [8]:
clean = df.copy()

# --- Fix 3: standardize fat labels ---
clean['Item_Fat_Content'] = clean['Item_Fat_Content'].replace(
    {'LF': 'Low Fat', 'low fat': 'Low Fat', 'reg': 'Regular'})

# --- Derive item category from SKU prefix (FD / DR / NC) ---
clean['Item_Category'] = clean['Item_Identifier'].str[:2].map(
    {'FD': 'Food', 'DR': 'Drinks', 'NC': 'Non-Consumable'})

# --- Fix 4: non-consumables cannot have a fat class ---
clean.loc[clean['Item_Category'] == 'Non-Consumable', 'Item_Fat_Content'] = 'Non-Edible'

print(clean['Item_Fat_Content'].value_counts().to_dict())
print(clean['Item_Category'].value_counts().to_dict())

{'Low Fat': 3918, 'Regular': 3006, 'Non-Edible': 1599}
{'Food': 6125, 'Non-Consumable': 1599, 'Drinks': 799}


In [9]:
# --- Fix 1: impute Item_Weight per SKU (same SKU = same weight), fallback Item_Type mean ---
clean['Item_Weight'] = clean['Item_Weight'].fillna(
    clean.groupby('Item_Identifier')['Item_Weight'].transform('mean'))
clean['Item_Weight'] = clean['Item_Weight'].fillna(
    clean.groupby('Item_Type')['Item_Weight'].transform('mean'))

print('Weight missing after imputation:', clean.Item_Weight.isna().sum())

Weight missing after imputation: 0


In [10]:
# --- Fix 2: fill Outlet_Size for OUT010 / OUT017 / OUT045 ---
clean['Outlet_Size'] = clean['Outlet_Size'].fillna('Small')

print('Size missing after fill:', clean.Outlet_Size.isna().sum())
print(clean.Outlet_Size.value_counts().to_dict())

Size missing after fill: 0
{'Small': 4798, 'Medium': 2793, 'High': 932}


In [11]:
# --- Fix 5: visibility 0 -> NaN -> per-SKU mean, fallback overall median ---
clean['Item_Visibility'] = clean['Item_Visibility'].replace(0, np.nan)
clean['Item_Visibility'] = clean['Item_Visibility'].fillna(
    clean.groupby('Item_Identifier')['Item_Visibility'].transform('mean'))
clean['Item_Visibility'] = clean['Item_Visibility'].fillna(clean['Item_Visibility'].median())

print('Zero visibility after fix:', (clean.Item_Visibility == 0).sum())
print('Missing visibility:', clean.Item_Visibility.isna().sum())

Zero visibility after fix: 0
Missing visibility: 0


In [12]:
# --- Enrichment columns for Power BI ---
clean['Outlet_Age'] = 2026 - clean['Outlet_Establishment_Year']
clean['MRP_Band'] = pd.cut(clean['Item_MRP'], [0, 70, 130, 200, 270],
                           labels=['Low (<70)', 'Mid (70-130)', 'High (130-200)', 'Premium (>200)'])

clean.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,Item_Category,Outlet_Age,MRP_Band
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380,Food,27,Premium (>200)
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228,Drinks,17,Low (<70)
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700,Food,27,High (130-200)
3,FDX07,19.20,Regular,0.022911,Fruits and Vegetables,182.0950,OUT010,1998,Small,Tier 3,Grocery Store,732.3800,Food,28,High (130-200)
4,NCD19,8.93,Non-Edible,0.016164,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052,Non-Consumable,39,Low (<70)


In [13]:
# --- Final validation: zero missing, totals unchanged ---
assert clean.isna().sum().sum() == 0, 'Still has missing values!'
assert len(clean) == len(df), 'Row count changed!'
assert abs(clean.Item_Outlet_Sales.sum() - df.Item_Outlet_Sales.sum()) < 0.01, 'Sales total changed!'

print('All checks passed')
print('Rows:', len(clean), '| Total Sales:', round(clean.Item_Outlet_Sales.sum(), 2))
clean.isna().sum()

All checks passed
Rows: 8523 | Total Sales: 18591125.41


Item_Identifier              0
Item_Weight                  0
Item_Fat_Content             0
Item_Visibility              0
Item_Type                    0
Item_MRP                     0
Outlet_Identifier            0
Outlet_Establishment_Year    0
Outlet_Size                  0
Outlet_Location_Type         0
Outlet_Type                  0
Item_Outlet_Sales            0
Item_Category                0
Outlet_Age                   0
MRP_Band                     0
dtype: int64

In [14]:
# --- Export clean dataset for Power BI ---
clean.to_excel('Blinkit_Grocery_Clean.xlsx', index=False)
clean.to_csv('Blinkit_Grocery_Clean.csv', index=False)
print('Saved: Blinkit_Grocery_Clean.xlsx / .csv')

Saved: Blinkit_Grocery_Clean.xlsx / .csv
